# SQL Gym — 02: Window Functions

Practice: `OVER`, `PARTITION BY`, frame specifications, ranking functions, `LAG`/`LEAD`, and running aggregates.
Write your SQL in the `%%solution N` cell and run it — results preview inline. Then run the check cell to validate.

**Tables:** `users`, `accounts`, `merchants`, `transactions`

In [1]:
from pathlib import Path
import sys

_cwd = Path.cwd()
_candidates = [_cwd / "sql", _cwd, _cwd.parent, _cwd.parent / "sql",
               _cwd.parent.parent, _cwd.parent.parent / "sql"]
_sql_dir = next((p for p in _candidates if (p / "utils" / "__init__.py").exists()), None)
if _sql_dir is None:
    raise RuntimeError(
        "Cannot locate sql/utils. Run: uv run jupyter lab from the project root."
    )
if str(_sql_dir) not in sys.path:
    sys.path.insert(0, str(_sql_dir))

DATA_DIR = _sql_dir / "data"

from utils import get_conn, check, register_sql_magic
from utils.checks.window_functions import Checker

conn = get_conn(DATA_DIR)
checker = Checker(conn)
register_sql_magic()
print("Ready. Tables: users, merchants, accounts, transactions")

Ready. Tables: users, merchants, accounts, transactions


In [2]:
for table in ["users", "merchants", "accounts", "transactions"]:
    print(f"\n{'─'*50}\n  {table}\n{'─'*50}")
    display(conn.execute(f"SELECT * FROM {table} LIMIT 3").df())
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  ({n:,} rows total)")


──────────────────────────────────────────────────
  users
──────────────────────────────────────────────────


,user_id,name,email,country,tier,kyc_verified,created_date
0,1,Carlos Clark,user1@example.com,IN,basic,True,2021-02-09
1,2,Tara Hernandez,user2@example.com,SG,basic,False,2022-12-11
2,3,Marcus Robinson,user3@example.com,AU,basic,True,2020-02-15


  (500 rows total)

──────────────────────────────────────────────────
  merchants
──────────────────────────────────────────────────


,merchant_id,name,mcc_category,city,country
0,1,DailyBasket Groceries 1,groceries,Austin,UK
1,2,GreenLeaf Groceries 2,groceries,New York,IN
2,3,DailyBasket Groceries 3,groceries,Chicago,UK


  (200 rows total)

──────────────────────────────────────────────────
  accounts
──────────────────────────────────────────────────


,account_id,user_id,account_type,opened_date,balance,status
0,1,1,checking,2021-02-21,13714.20,active
1,2,2,savings,2023-01-21,3240.81,active
2,3,3,credit,2020-07-19,5271.51,active


  (600 rows total)

──────────────────────────────────────────────────
  transactions
──────────────────────────────────────────────────


,txn_id,account_id,merchant_id,amount,txn_type,txn_date,status
0,1,571,136,18.77,debit,2024-09-21,completed
1,2,21,124,17.97,debit,2024-12-19,completed
2,3,564,165,198.60,debit,2022-10-09,completed


  (20,000 rows total)


## Problem 1: Running Balance per Account

For each completed transaction, compute the running balance of the account. Treat debits as negative and credits as positive. Order by account_id, txn_date, txn_id within each partition.

<details>
<summary>Hint</summary>

Use `SUM(CASE WHEN txn_type = 'credit' THEN amount ELSE -amount END) OVER (PARTITION BY account_id ORDER BY txn_date, txn_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`. The `ROWS BETWEEN` frame spec makes this a true running (not sliding) total. Filter `status = 'completed'` before windowing.

</details>

| Column | Type | Notes |
|--------|------|-------|
| txn_id | integer | |
| account_id | integer | |
| txn_date | date | |
| amount | double | |
| txn_type | string | |
| running_balance | double | 2 decimal places, cumulative signed sum |

Expected: all completed transactions ordered by account_id, txn_date, txn_id.

In [3]:
%%solution 1

SELECT
txn_id,
account_id,
txn_date,
amount,
txn_type,
ROUND(
    SUM(CASE WHEN txn_type = 'debit' THEN -amount ELSE amount END) 
    OVER ( 
        PARTITION BY account_id 
        ORDER BY (txn_date, txn_id) 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) 
AS running_balance
FROM transactions txn
WHERE txn.status = 'completed'
ORDER BY account_id, txn_date, txn_id

,txn_id,account_id,txn_date,amount,txn_type,running_balance
0,11142,1,2022-03-18,42.93,debit,-42.93
1,12147,1,2022-06-29,33.26,debit,-76.19
2,12072,1,2022-10-25,1.00,debit,-77.19
3,2484,1,2022-11-14,9.19,debit,-86.38
4,9920,1,2022-12-03,197.15,debit,-283.53
...,...,...,...,...,...,...
16028,429,600,2024-03-06,12.93,credit,-602.92
16029,8725,600,2024-06-12,172.37,debit,-775.29
16030,16973,600,2024-08-14,2.54,debit,-777.83
16031,11608,600,2024-11-23,95.13,credit,-682.70


In [4]:
checker.p1(solution_1)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 2: Top 3 Merchants by Revenue per Category

For each MCC category, rank merchants by their total completed revenue and return the top 3. Use `DENSE_RANK` so ties are handled correctly.

<details>
<summary>Hint</summary>

First aggregate revenue per merchant in a CTE. Then apply `DENSE_RANK() OVER (PARTITION BY mcc_category ORDER BY total_revenue DESC)`. Use `QUALIFY revenue_rank <= 3` to filter in DuckDB/Snowflake. In PostgreSQL/Redshift, wrap in a subquery or CTE and filter there. Note: `QUALIFY` is shorthand — both approaches are valid in interviews.

</details>

| Column | Type | Notes |
|--------|------|-------|
| merchant_id | integer | |
| name | string | |
| mcc_category | string | |
| total_revenue | double | 2 decimal places |
| revenue_rank | bigint | rank within category |

Expected: up to 3 rows per category (8 categories = ≤24 rows total), ordered by mcc_category ASC, revenue_rank ASC.

In [5]:
%%solution 2

SELECT
mr.merchant_id,
name,
mcc_category,
SUM(amount) as total_revenue,
DENSE_RANK() OVER (PARTITION BY mcc_category ORDER BY total_revenue DESC) revenue_rank
FROM transactions txn
JOIN merchants mr
ON txn.merchant_id = mr.merchant_id
WHERE txn.status = 'completed'
GROUP BY 1, 2, 3
QUALIFY revenue_rank <= 3
ORDER BY mcc_category, revenue_rank

,merchant_id,name,mcc_category,total_revenue,revenue_rank
0,90,CinePlex Entertainment 15,entertainment,30594.20,1
1,92,CinePlex Entertainment 17,entertainment,21113.00,2
2,100,LiveBeat Entertainment 25,entertainment,16369.70,3
3,185,DriveGo Fuel 10,fuel,50802.85,1
4,187,GasPro Fuel 12,fuel,23572.22,2
5,193,GasPro Fuel 18,fuel,16576.33,3
6,22,DailyBasket Groceries 22,groceries,43833.58,1
7,5,FreshMart Groceries 5,groceries,39546.39,2
8,10,DailyBasket Groceries 10,groceries,37877.87,3
9,169,PharmaDirect Healthcare 19,healthcare,34160.76,1


In [6]:
checker.p2(solution_2)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 3: Month-over-Month Spend Change by Country

For each country, compute monthly total debit spending and the month-over-month percentage change. Null is expected for the first month of each country.

<details>
<summary>Hint</summary>

Use a CTE to compute monthly spend per country. In the outer query, use `LAG(monthly_spend, 1) OVER (PARTITION BY country ORDER BY month)` to access the previous month. Percentage change: `(current - previous) / previous * 100`. Handle nulls — the first row per partition will have `NULL` prev value.

</details>

| Column | Type | Notes |
|--------|------|-------|
| country | string | |
| month | date | monthly bucket |
| monthly_spend | double | 2 decimal places |
| prev_month_spend | double | 2 decimal places, NULL for first month |
| mom_pct_change | double | %, 2 decimal places, NULL for first month |

Expected: rows ordered by country ASC, month ASC.

In [7]:
%%solution 3

SELECT
country,
DATE_TRUNC('month', txn_date) AS month,
ROUND(SUM(txn.amount), 2) AS monthly_spend,
LAG(monthly_spend, 1, Null) OVER (PARTITION BY country ORDER BY month) AS prev_month_spend,
ROUND( (monthly_spend - prev_month_spend) / prev_month_spend * 100 , 2) as mom_pct_change
FROM transactions txn
JOIN accounts acc
ON txn.account_id = acc.account_id
JOIN users
ON acc.user_id = users.user_id
WHERE txn.txn_type = 'debit' AND txn.status = 'completed'
GROUP BY 1, 2
ORDER BY 1, 2

,country,month,monthly_spend,prev_month_spend,mom_pct_change
0,AU,2022-01-01,6269.02,NaN,NaN
1,AU,2022-02-01,3179.62,6269.02,-49.28
2,AU,2022-03-01,7240.25,3179.62,127.71
3,AU,2022-04-01,7599.58,7240.25,4.96
4,AU,2022-05-01,8777.75,7599.58,15.50
...,...,...,...,...,...
175,US,2024-08-01,19978.27,20308.27,-1.62
176,US,2024-09-01,14530.21,19978.27,-27.27
177,US,2024-10-01,18265.06,14530.21,25.70
178,US,2024-11-01,20047.08,18265.06,9.76


In [8]:
checker.p3(solution_3)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 4: Top 2 Users by Spend per Country

Within each country, find the top 2 users by total completed debit spending. Use `ROW_NUMBER` so each country has exactly 2 rows (no ties in ranking).

<details>
<summary>Hint</summary>

Use a CTE to aggregate spend per user. Apply `ROW_NUMBER() OVER (PARTITION BY country ORDER BY total_spend DESC)`. Then `QUALIFY spend_rank <= 2`. The difference from DENSE_RANK: ROW_NUMBER gives unique ranks even for ties, ensuring exactly 2 per country.

</details>

| Column | Type | Notes |
|--------|------|-------|
| user_id | integer | |
| name | string | |
| country | string | |
| tier | string | |
| total_spend | double | 2 decimal places |
| spend_rank | bigint | rank within country |

Expected: **10 rows** (2 per country × 5 countries), ordered by country ASC, spend_rank ASC.

In [9]:
%%solution 4

SELECT
acc.user_id,
name,
country,
tier,
ROUND(SUM(amount), 2) AS total_spend,
ROW_NUMBER() OVER (PARTITION BY country ORDER BY total_spend DESC) AS spend_rank
FROM transactions txn
JOIN accounts acc ON txn.account_id = acc.account_id
JOIN users ON acc.user_id = users.user_id
WHERE txn.status = 'completed' AND txn.txn_type = 'debit'
GROUP BY 1, 2, 3, 4
QUALIFY spend_rank <= 2
ORDER BY country, spend_rank

,user_id,name,country,tier,total_spend,spend_rank
0,465,Marcus Thompson,AU,premium,48599.39,1
1,168,Nina Lewis,AU,premium,37658.14,2
2,74,Bob Davis,IN,business,34609.94,1
3,423,Alice Smith,IN,basic,17131.80,2
4,21,Ethan Harris,SG,business,34691.82,1
5,428,Ivan Robinson,SG,basic,33514.62,2
6,332,Bob Miller,UK,basic,29856.06,1
7,217,Yusuf Brown,UK,basic,29128.60,2
8,473,Diana Brown,US,basic,40056.44,1
9,52,Yusuf Smith,US,basic,35874.99,2


In [10]:
checker.p4(solution_4)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 5: Percentile Rank of Transaction Amounts

For each completed transaction, compute its percentile rank within its MCC category. A value of 0.0 is the lowest; 1.0 is the highest.

<details>
<summary>Hint</summary>

`PERCENT_RANK() OVER (PARTITION BY mcc_category ORDER BY amount)` — returns a float in [0.0, 1.0]. No CTE needed. Filter `status = 'completed'`. This is identical in DuckDB, Snowflake, BigQuery, and PostgreSQL.

</details>

| Column | Type | Notes |
|--------|------|-------|
| txn_id | integer | |
| mcc_category | string | |
| amount | double | |
| pct_rank | double | 0.0 (lowest) to 1.0 (highest), 4 decimal places |

Expected: all completed transactions, ordered by mcc_category ASC, amount ASC.

In [11]:
%%solution 5

SELECT
txn_id,
mcc_category,
amount,
ROUND(PERCENT_RANK() OVER (PARTITION BY mcc_category ORDER BY amount), 4) AS pct_rank
FROM transactions txn
JOIN merchants mr ON txn.merchant_id = mr.merchant_id
WHERE txn.status = 'completed'
ORDER BY mcc_category, amount

,txn_id,mcc_category,amount,pct_rank
0,10882,entertainment,1.00,0.0000
1,14581,entertainment,1.00,0.0000
2,17060,entertainment,1.00,0.0000
3,11749,entertainment,1.00,0.0000
4,16886,entertainment,1.12,0.0022
...,...,...,...,...
16028,18561,utilities,4235.87,0.9983
16029,10810,utilities,6558.65,0.9987
16030,14452,utilities,7601.94,0.9992
16031,10081,utilities,9750.63,0.9996


In [12]:
checker.p5(solution_5)  # type: ignore[name-defined]  # noqa: F821

True